# 🤺 Fencing AI — Train the model (free, in your browser)

Runs the whole pipeline on **Google Colab** — no PC needed, works from a tablet.

**Pipeline:** download FIE videos → MediaPipe pose → Claude labels → train → export `sabre.onnx`.

**Cost:** Colab is free. The only cost is Claude labeling (defaults to cheap Haiku, ~$25 for ~50 videos). For $0, skip labeling and hand-label in the app's `/dataset` page instead.

Run each cell in order (Shift+Enter).

> **Tip:** Runtime → Change runtime type → **T4 GPU** (free) for faster training, though CPU works fine too.

## 1. Get the code

If your GitHub repo is **private**, create a token at https://github.com/settings/tokens (classic, `repo` scope) and paste it when prompted. If it's public, just press Enter.

In [ ]:
from getpass import getpass
import os

REPO = "crazymelol/thementalsport.com"
token = getpass("GitHub token (press Enter if repo is public): ").strip()

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
!rm -rf thementalsport.com
!git clone --depth 1 --branch claude/new-project-BzhH4 {url}
os.chdir("thementalsport.com/fencing-ai/backend")
print("\nNow in:", os.getcwd())

## 2. Install dependencies (~2 min)

In [ ]:
!pip install -q -r requirements.txt onnxruntime
print("✓ dependencies installed")

## 3. Your Anthropic API key

Get one at https://console.anthropic.com/settings/keys. It's kept in memory only — not saved to the notebook.

In [ ]:
os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (sk-ant-...): ").strip()
# Cheapest labeling model by default. For best labels: os.environ['ANALYZER_MODEL'] = 'claude-opus-4-8'
os.environ.setdefault("ANALYZER_MODEL", "claude-haiku-4-5-20251001")
print("✓ key set, labeling model:", os.environ["ANALYZER_MODEL"])

## 4. Paste your FIE bout URLs

From the official channel: https://www.youtube.com/@FIEvideo — open sabre bouts and copy the URLs. Add as many as you like (50+ recommended).

In [ ]:
urls = """
https://www.youtube.com/watch?v=REPLACE_ME_1
https://www.youtube.com/watch?v=REPLACE_ME_2
""".strip()

with open("dataset/urls.txt", "w") as f:
    f.write(urls)
print("Saved", len([u for u in urls.splitlines() if u.strip()]), "URLs")

## 5. Build the dataset (Claude auto-labels every video)

This downloads each video, runs pose estimation, and labels actions. Resumable — re-run to add more videos. *This is the step that uses Claude API credits.*

In [ ]:
!python -m dataset.build_dataset dataset/urls.txt

## 6. Train the model

In [ ]:
!python -m training.train_action_classifier --epochs 30

## 7. Export to ONNX for the live judge

In [ ]:
!python -m training.export_onnx --out sabre.onnx
print("\nFiles produced: sabre.onnx + sabre.json")

## 8. Download the trained model

Save these two files, then drop them into `fencing-ai/frontend/public/models/` (commit + push, or upload in your host's dashboard). Reload `/live` — the badge flips to **AI MODEL**.

In [ ]:
from google.colab import files
files.download("sabre.onnx")
files.download("sabre.json")